In [70]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [71]:
DATA_PATH = Path("../data/raw")

## Dataset Assessment

### Objective

This notebook evaluates whether the Olist dataset is suitable for building the InsightFlow Customer Analytics Platform.

Unlike the profiling stage, this notebook focuses on investigating data quality issues that may affect downstream analytics, database design, and machine learning.

The outcome of this notebook serves as evidence for technical decisions documented in the Decision Log (D002, D003, ...).

---

### Assessment Scope

- Order Timeline Consistency
- Payment Validation
- Dataset Readiness Summary

In [75]:
orders = pd.read_csv(
    DATA_PATH / "olist_orders_dataset.csv"
)

payments = pd.read_csv(
    DATA_PATH / "olist_order_payments_dataset.csv"
)

order_items = pd.read_csv(
    DATA_PATH / "olist_order_items_dataset.csv"
)
customers = pd.read_csv(
    DATA_PATH / "olist_customers_dataset.csv"
)
products = pd.read_csv(
    DATA_PATH / "olist_products_dataset.csv"
)
sellers = pd.read_csv(
    DATA_PATH / "olist_sellers_dataset.csv"
)
reviews = pd.read_csv(
    DATA_PATH / "olist_order_reviews_dataset.csv"
)

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print("Datetime conversion completed.")

Datetime conversion completed.


## Section 1 — Order Timeline Assessment

Objective

Evaluate chronological consistency between order timestamps to determine whether timeline anomalies are severe enough to affect downstream analytics.

Decision Output

D002 — Order Timeline Strategy

## Timeline Rule Validation

### Objective

Validate whether the chronological order of order timestamps follows the expected business process.

Expected order lifecycle:

Purchase
→ Approved
→ Handed to Carrier
→ Delivered to Customer

Any violation of this sequence will be investigated further to determine its impact on downstream analytics.

In [ ]:
rule_purchase_approved = orders[
    orders["order_purchase_timestamp"] >
    orders["order_approved_at"]
].copy()

rule_approved_carrier = orders[
    orders["order_approved_at"] >
    orders["order_delivered_carrier_date"]
].copy()

rule_carrier_customer = orders[
    orders["order_delivered_carrier_date"] >
    orders["order_delivered_customer_date"]
].copy()

rule_purchase_customer = orders[
    orders["order_purchase_timestamp"] >
    orders["order_delivered_customer_date"]
].copy()

timeline_summary = pd.DataFrame({
    "Rule": [
        "Purchase > Approved",
        "Approved > Carrier",
        "Carrier > Customer",
        "Purchase > Customer"
    ],
    "Violation Count": [
        len(rule_purchase_approved),
        len(rule_approved_carrier),
        len(rule_carrier_customer),
        len(rule_purchase_customer)
    ]
})

timeline_summary["Percentage (%)"] = (
    timeline_summary["Violation Count"] /
    len(orders) * 100
).round(2)

timeline_summary

,Rule,Violation Count,Percentage (%)
0,Purchase > Approved,0,0.00
1,Approved > Carrier,1359,1.37
2,Carrier > Customer,23,0.02
3,Purchase > Customer,0,0.00


### Observation

- A total of **1,359** records (**1.37%**) have timestamp sequence inconsistencies between `order_approved_at` and `order_delivered_carrier_date`.
- A total of **23** records (**0.02%**) have timestamp sequence inconsistencies between `order_delivered_carrier_date` and `order_delivered_customer_date`.
- No violations were identified in the `purchase → approved` or `purchase → customer` relationships.

## Unique Order Analysis

### Objective

Determine whether timeline anomalies originate from unique orders or whether the same order violates multiple chronological rules.

In [21]:
all_anomalies = pd.concat([
    rule_purchase_approved,
    rule_approved_carrier,
    rule_carrier_customer,
    rule_purchase_customer
])

total_anomaly_rows = len(all_anomalies)

unique_orders = all_anomalies["order_id"].nunique()

print("Unique Order Investigation")
print("-"*50)

print(f"Total anomaly rows : {total_anomaly_rows:,}")
print(f"Unique order_id    : {unique_orders:,}")
print(f"Total orders       : {len(orders):,}")
print(f"Percentage         : {unique_orders / len(orders) * 100:.2f}%")

Unique Order Investigation
--------------------------------------------------
Total anomaly rows : 1,382
Unique order_id    : 1,382
Total orders       : 99,441
Percentage         : 1.39%


### Observation

- A total of **1,382** unique orders (**1.39%** of **99,441** orders) were identified with timestamp sequence inconsistencies.
- The number of anomaly rows and unique `order_id` values are both **1,382**, indicating that each order identified as an anomaly violated only one timeline rule.
- The inconsistencies were primarily found between `order_approved_at` and `order_delivered_carrier_date`, affecting **1,359** orders (**1.37%**), while **23** orders (**0.02%**) had inconsistencies between `order_delivered_carrier_date` and `order_delivered_customer_date`.

## Multiple Rule Violations

### Objective

Assess whether timeline anomalies are isolated events or whether individual orders violate multiple chronological rules.

In [20]:
violation_frequency = (
    all_anomalies["order_id"]
    .value_counts()
)

multiple_rule_summary = (
    violation_frequency
    .value_counts()
    .sort_index()
    .rename_axis("Rules Violated")
    .reset_index(name="Number of Orders")
)

multiple_rule_summary

,Rules Violated,Number of Orders
0,1,1382


### Observation

- All **1,382** orders identified with timeline inconsistencies violated only one timeline rule, indicating that no order was found to violate more than one timestamp sequence rule.

## Timeline Severity Assessment

### Objective

Measure the magnitude of timeline inconsistencies.

The purpose of this assessment is to distinguish between minor timestamp inconsistencies (e.g., minutes) and severe anomalies (e.g., days), since both may require different handling strategies.

In [ ]:
carrier_anomaly = orders.loc[
    orders["order_delivered_carrier_date"] <
    orders["order_approved_at"]
].copy()

carrier_anomaly["difference"] = (
    carrier_anomaly["order_approved_at"] -
    carrier_anomaly["order_delivered_carrier_date"]
)

carrier_anomaly["difference_hours"] = (
    carrier_anomaly["difference"].dt.total_seconds() / 3600
)

print("Time Difference Analysis : Approved vs Carrier")
print("-" * 50)

print(f"Anomaly orders : {len(carrier_anomaly):,} baris")
print()

print("Difference (hours):")
print(f"Minimum        : {carrier_anomaly['difference_hours'].min():.2f} jam")
print(f"25%            : {carrier_anomaly['difference_hours'].quantile(0.25):.2f} jam")
print(f"Median         : {carrier_anomaly['difference_hours'].median():.2f} jam")
print(f"Mean           : {carrier_anomaly['difference_hours'].mean():.2f} jam")
print(f"75%            : {carrier_anomaly['difference_hours'].quantile(0.75):.2f} jam")
print(f"Maximum        : {carrier_anomaly['difference_hours'].max():.2f} jam")

Time Difference Analysis : Approved vs Carrier
--------------------------------------------------
Anomaly orders : 1,359 baris

Difference (hours):
Minimum        : 0.01 jam
25%            : 1.42 jam
Median         : 17.17 jam
Mean           : 24.75 jam
75%            : 25.96 jam
Maximum        : 4109.26 jam


### Observation

- Among the **1,359** orders with inconsistencies between `order_approved_at` and `order_delivered_carrier_date`, the median time difference was **17.17 hours**.
- **75%** of the inconsistencies had a time difference of approximately **25.96 hours or less**, indicating that most inconsistencies were relatively small.
- However, an outlier with a time difference of approximately **171 days** was identified.
- The mean time difference of **24.75 hours** being higher than the median of **17.17 hours** indicates that the distribution was influenced by several outliers with substantially larger time differences.

## Temporal Distribution

### Objective

Determine whether timeline anomalies occur consistently throughout the observation period or are concentrated during specific time periods.

In [26]:
carrier_anomaly["year_month"] = (
    carrier_anomaly["order_purchase_timestamp"]
    .dt.to_period("M")
)

carrier_anomaly["year_month"] \
    .value_counts() \
    .sort_index()

monthly_anomaly = (
    carrier_anomaly["year_month"]
    .value_counts()
    .sort_index()
)

print("Monthly Distribution : Approved vs Carrier Anomaly")
print("-" * 50)

print(f"Total anomaly orders : {len(carrier_anomaly):,} baris")
print()

for month, count in monthly_anomaly.items():
    print(f"{str(month):<10} : {count:>5,} baris")

Monthly Distribution : Approved vs Carrier Anomaly
--------------------------------------------------
Total anomaly orders : 1,359 baris

2017-02    :     1 baris
2017-04    :    24 baris
2017-05    :    14 baris
2017-06    :     4 baris
2017-07    :    21 baris
2017-08    :     1 baris
2017-09    :    13 baris
2017-11    :     1 baris
2017-12    :     5 baris
2018-01    :    25 baris
2018-02    :    12 baris
2018-03    :     2 baris
2018-04    :   393 baris
2018-05    :    68 baris
2018-06    :   123 baris
2018-07    :   566 baris
2018-08    :    86 baris


### Observation

- The inconsistency between `order_approved_at` and `order_delivered_carrier_date` was not evenly distributed throughout the observation period and was concentrated primarily in **2018**.
- The highest number of inconsistencies occurred in **July 2018**, with **566** orders, followed by **April 2018** with **393** orders and **June 2018** with **123** orders.
- These three months accounted for approximately **79.6%** of the **1,359** timestamp inconsistencies, indicating that the inconsistencies were concentrated in specific periods.
- The concentration of inconsistencies in these months should be further examined during the data cleaning stage to identify potential causes and determine an appropriate handling method.

## Order Status Distribution

### Objective

Analyze the distribution of order statuses among timeline anomalies.

This assessment aims to determine whether anomalous orders still represent valid business transactions (e.g., delivered orders) or are primarily associated with cancelled or unavailable orders.

The findings will help evaluate whether these records should be retained for downstream business analytics.

In [ ]:
status_distribution = (
    orders[
        orders["order_id"].isin(
            all_anomalies["order_id"]
        )
    ]["order_status"]
    .value_counts()
    .rename_axis("Order Status")
    .reset_index(name="Count")
)

status_distribution

,Order Status,Count
0,delivered,1373
1,shipped,9


### Observation

- Among the **1,382** orders with timestamp inconsistencies, **1,373** orders had a `delivered` status, while **9** orders had a `shipped` status.
- The majority of timestamp inconsistencies (**99.35%**) occurred in `delivered` orders, while **0.65%** occurred in `shipped` orders.

## Missing Timestamp Assessment

### Objective

Assess whether timeline anomalies are caused by missing timestamp values rather than incorrect chronological order.

This investigation helps distinguish between incomplete records and genuine temporal inconsistencies.

In [27]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].isna().sum().to_frame("Missing Values")

,Missing Values
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


### Observation

- `order_purchase_timestamp` and `order_estimated_delivery_date` have no missing values, indicating that both columns contain complete records.
- A total of **160** missing values were identified in `order_approved_at`, **1,783** in `order_delivered_carrier_date`, and **2,965** in `order_delivered_customer_date`.
- The number of missing values tends to increase at later stages of the order process, particularly in `order_delivered_carrier_date` and `order_delivered_customer_date`.
- Missing values in delivery-related timestamps should be considered because they may affect delivery duration analysis and timestamp sequence consistency checks.

## Revenue Impact Assessment

### Objective

Estimate the business impact of timeline anomalies by measuring their contribution to total revenue.

This assessment evaluates whether removing or excluding anomalous orders would materially affect business KPIs.

In [29]:
anomaly_orders = all_anomalies["order_id"].unique()

anomaly_revenue = (
    payments[
        payments["order_id"].isin(anomaly_orders)
    ]["payment_value"]
    .sum()
)

total_revenue = payments["payment_value"].sum()

print(f"Payment Value Impact : {(anomaly_revenue / total_revenue) * 100:.2f}%")

Payment Value Impact : 1.31%


### Observation

- Orders with timestamp inconsistencies accounted for approximately **1.31%** of the total `payment_value`, indicating that their financial impact on overall recorded transaction value was relatively limited.

## Timeline Severity Assessment

### Objective

Measure the magnitude of each timeline anomaly by quantifying the time difference between related events.

Rather than only counting the number of anomalies, this assessment evaluates how severe each inconsistency is. Comparing the severity of both anomaly types helps determine whether they represent minor system timestamp inconsistencies or more substantial data quality issues.

The findings from this section provide additional evidence for selecting an appropriate handling strategy in **D002 – Order Timeline Strategy**.

In [38]:
# Approved -> Carrier
# ==========================================================

carrier_anomaly = orders.loc[
    orders["order_delivered_carrier_date"] <
    orders["order_approved_at"]
].copy()

carrier_anomaly["difference_hours"] = (
    carrier_anomaly["order_approved_at"] -
    carrier_anomaly["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print("Severity Analysis : Approved → Carrier")
print("-" * 70)

carrier_anomaly["difference_hours"].describe()

Severity Analysis : Approved → Carrier
----------------------------------------------------------------------


count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
max      4109.256111
Name: difference_hours, dtype: float64

In [39]:
# Carrier -> Customer
# ==========================================================

customer_anomaly = orders.loc[
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
].copy()

customer_anomaly["difference_hours"] = (
    customer_anomaly["order_delivered_carrier_date"] -
    customer_anomaly["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print("Severity Analysis : Carrier → Customer")
print("-" * 50)

customer_anomaly["difference_hours"].describe()

Severity Analysis : Carrier → Customer
--------------------------------------------------


count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: difference_hours, dtype: float64

In [34]:
severity_summary = pd.DataFrame({
    "Anomaly Type": [
        "Approved → Carrier",
        "Carrier → Customer"
    ],
    "Count": [
        len(carrier_anomaly),
        len(customer_anomaly)
    ],
    "Median (Hours)": [
        carrier_anomaly["difference_hours"].median(),
        customer_anomaly["difference_hours"].median()
    ],
    "Mean (Hours)": [
        carrier_anomaly["difference_hours"].mean(),
        customer_anomaly["difference_hours"].mean()
    ],
    "Maximum (Hours)": [
        carrier_anomaly["difference_hours"].max(),
        customer_anomaly["difference_hours"].max()
    ]
})

severity_summary.round(2)

,Anomaly Type,Count,Median (Hours),Mean (Hours),Maximum (Hours)
0,Approved → Carrier,1359,17.17,24.75,4109.26
1,Carrier → Customer,23,39.86,78.46,386.31


In [37]:
print("Order Status Distribution")
print("-" * 30)

print("\nApproved → Carrier")
display(
    carrier_anomaly["order_status"]
    .value_counts()
    .to_frame("Count")
)

print("\nCarrier → Customer")
display(
    customer_anomaly["order_status"]
    .value_counts()
    .to_frame("Count")
)

Order Status Distribution
------------------------------

Approved → Carrier


,Count
order_status,
delivered,1350
shipped,9



Carrier → Customer


,Count
order_status,
delivered,23


### Observation

- A total of **1,382** unique orders were identified with timestamp sequence inconsistencies, representing approximately **1.39%** of the **99,441** orders.
- The most frequent inconsistency occurred in the `order_approved_at` → `order_delivered_carrier_date` sequence, affecting **1,359** orders (**1.37%**). The median time difference was **17.17 hours**, with a mean of **24.75 hours** and a maximum of **4,109.26 hours**, equivalent to approximately **171 days**.
- The `order_delivered_carrier_date` → `order_delivered_customer_date` sequence contained inconsistencies in **23** orders (**0.02%**). The median time difference was **39.86 hours**, with a mean of **78.46 hours** and a maximum of **386.31 hours**, equivalent to approximately **16.1 days**.
- For the `Approved → Carrier` inconsistency, **1,350** orders had a `delivered` status and **9** had a `shipped` status. Meanwhile, all **23** orders with the `Carrier → Customer` inconsistency had a `delivered` status.
- Timestamp inconsistencies were concentrated among orders with a `delivered` status, suggesting that the issue is more related to timestamp recording consistency in completed orders than simply orders that are still in the delivery process.
- Based on the previous temporal distribution analysis, `Approved → Carrier` inconsistencies were also concentrated during certain periods, particularly from **April to August 2018**, with the highest number occurring in **July 2018**, with **566** orders.
- Orders with timestamp inconsistencies accounted for approximately **1.31%** of the total `payment_value`, indicating that their impact on overall recorded transaction value was relatively limited.
- The comparison of delivery duration shows that orders with timestamp inconsistencies had an average delivery duration of **7.95 days**, compared with **12.15 days** for orders without inconsistencies. This difference is exploratory and should not be interpreted as a causal relationship.

## Business Impact Assessment

### Objective

Evaluate whether timeline anomalies have a material impact on key business metrics and downstream analytics.

This assessment compares anomalous and non-anomalous orders to determine whether the identified timeline inconsistencies meaningfully affect business analysis, or whether they represent isolated data quality issues with minimal analytical impact.

The findings from this assessment provide the final evidence for selecting an appropriate handling strategy in **D002 – Order Timeline Strategy**.

In [31]:
normal_orders = orders[
    ~orders["order_id"].isin(all_anomalies["order_id"])
].copy()

normal_orders["delivery_days"] = (
    normal_orders["order_delivered_customer_date"] -
    normal_orders["order_purchase_timestamp"]
).dt.days

anomaly_orders = orders[
    orders["order_id"].isin(all_anomalies["order_id"])
].copy()

anomaly_orders["delivery_days"] = (
    anomaly_orders["order_delivered_customer_date"] -
    anomaly_orders["order_purchase_timestamp"]
).dt.days

summary = pd.DataFrame({
    "Group":[
        "Normal",
        "Timeline Anomaly"
    ],
    "Average Delivery Days":[
        normal_orders["delivery_days"].mean(),
        anomaly_orders["delivery_days"].mean()
    ],
    "Median Delivery Days":[
        normal_orders["delivery_days"].median(),
        anomaly_orders["delivery_days"].median()
    ]
})

summary

,Group,Average Delivery Days,Median Delivery Days
0,Normal,12.153844,10.0
1,Timeline Anomaly,7.954843,7.0


### Observation

- Orders without timeline inconsistencies have an average delivery duration of **12.15 days** and a median of **10 days**.
- In comparison, orders with timeline inconsistencies have an average delivery duration of **7.95 days** and a median of **7 days**.
- Therefore, orders with timeline inconsistencies recorded a shorter delivery duration than orders without inconsistencies, with an average difference of approximately **4.20 days**.

## Summary of Findings Section 1

### Scope of the Issue

A total of **1,382 unique orders (1.39% of 99,441 total orders)** were found to have timestamp sequence inconsistencies, divided into two types: **1,359 orders (approved → carrier)** and **23 orders (carrier → customer)**. No order violated more than one rule simultaneously.

### Severity

The two types of anomalies show different magnitude patterns. The **approved → carrier** anomaly has a median time difference of **17.17 hours** (mean 24.75 hours, maximum 4,109 hours / ~171 days). The **carrier → customer** anomaly, although much less frequent (23 orders), has a higher median time difference of **39.86 hours** (mean 78.46 hours, maximum 386 hours / ~16 days). Both groups contain extreme outliers that pull the mean substantially away from the median, indicating **skewed distributions rather than uniformly distributed anomalies**.

### Pattern

The **approved → carrier** anomalies are heavily concentrated in **April, June, and July 2018**, accounting for **79.6% of all anomalies**. This concentration suggests a possible system or process disruption during these periods rather than random noise distributed evenly over time. Nearly all problematic orders (**99.35%**) were still marked as `delivered`, indicating that these were valid and completed transactions rather than failed or cancelled orders.

### Relevance to Business Question

The `order_purchase_timestamp` column, which serves as the primary timestamp for the **churn and repeat purchase analysis in InsightFlow**, is not affected by these anomalies. There were **0 violations** involving the purchase → approved and purchase → customer timestamp rules. Therefore, the anomalies are limited to **internal logistics process timestamps** (`approved`, `carrier`, and `delivered`) rather than the timestamps used to measure customer purchasing behavior.

### Delivery Performance Comparison

A comparison of average `delivery_days` (purchase → delivered customer) between normal orders and anomalous orders produced a result contrary to the initial expectation. Anomalous orders actually had a **shorter median delivery time of 7 days** compared with **10 days for normal orders**.

This finding suggests that anomalies in `order_approved_at` may represent **timestamp recording artifacts**, such as delayed status logging, rather than actual fulfillment problems. However, this interpretation remains a **hypothesis** and has not been formally verified. No statistical test was conducted to determine whether the difference in delivery time is statistically significant.

### Business Impact

Orders with timeline anomalies account for only **1.31% of the total `payment_value`**, representing a relatively small proportion with **limited material impact on overall revenue KPIs**.

### Implication for Decision

The relatively small scale of the anomalies (**1.39% of orders and 1.31% of revenue**), their lack of impact on the critical `order_purchase_timestamp` used in the main business question, and the fact that the affected orders remain valid based on their business status indicate **low risk in retaining these records in the dataset**.

However, the concentration of anomalies during **April–July 2018** should still be documented as a separate finding and may warrant further investigation if the project is expanded to include **operational or logistics performance analysis** in the future.

# Section 2 — Payment Validation Assessment

## Objective

Evaluate the validity and consistency of payment records to identify values that may negatively affect downstream analytics and business reporting.

This assessment focuses on detecting invalid or suspicious payment transactions, evaluating their potential business impact, and determining the most appropriate handling strategy before data preparation.

The findings from this section serve as the evidence for:

- **D003 — Invalid Payment Handling Strategy**

## Payment Value Validation

### Objective

Identify payment records containing invalid monetary values and quantify the overall magnitude of the issue.

This assessment determines the proportion of payment records with zero or negative payment values before evaluating their impact on downstream analytics.

In [41]:
invalid_payment = payments.loc[
    payments["payment_value"] <= 0
].copy()

summary = pd.DataFrame({
    "Condition": [
        "Payment <= 0",
        "Payment > 0"
    ],
    "Count": [
        len(invalid_payment),
        len(payments) - len(invalid_payment)
    ]
})

summary["Percentage (%)"] = (
    summary["Count"] /
    len(payments) * 100
).round(2)

summary

,Condition,Count,Percentage (%)
0,Payment <= 0,9,0.01
1,Payment > 0,103877,99.99


### Observation

- A total of **9** payment records have `payment_value` less than or equal to **0**, representing approximately **0.01%** of the **103,886** payment records.
- The remaining **103,877** payment records (**99.99%**) have `payment_value` greater than **0**.

## Payment Type Distribution

### Objective

Investigate whether invalid payment records are concentrated within specific payment methods.

Understanding the payment type distribution helps determine whether the issue is associated with a particular payment channel or represents isolated records.

In [ ]:
# Payment Type Investigation
# ==========================================================

payment_type_summary = (
    invalid_payment["payment_type"]
    .value_counts()
    .rename_axis("Payment Type")
    .reset_index(name="Count")
)

payment_type_summary

,Payment Type,Count
0,voucher,6
1,not_defined,3


### Observation

- Among the **9** payment records with `payment_value <= 0`, **6** records are associated with the `voucher` payment type, while **3** records are categorized as `not_defined`.
- The majority of non-positive payment records are therefore associated with `voucher` transactions, suggesting that zero-value payment records may be related to the use or recording of vouchers.
- The **3** `not_defined` records indicate that the payment type was not explicitly identified and require further investigation, particularly when considering their corresponding `order_status`.

## Order Status Assessment

### Objective

Evaluate the business outcome of orders associated with invalid payment records.

This assessment determines whether invalid payment values are primarily linked to completed, cancelled, or unavailable orders.

In [45]:
payment_status = (
    invalid_payment
    .merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
)

payment_status["order_status"] \
    .value_counts() \
    .to_frame("Count")

,Count
order_status,
delivered,4
canceled,3
shipped,2


### Observation

- Among the **9** payment records with `payment_value <= 0`, **4** are associated with `delivered` orders, **3** with `canceled` orders, and **2** with `shipped` orders.
- Non-positive payment values were found in both completed orders (`delivered`) and orders that were canceled or still in the shipping process.
- The presence of **3** payment records associated with `canceled` orders provides relevant business context, as order cancellations may be related to payment processing or transaction cancellation.
- Payment records with `payment_value <= 0` associated with `delivered` and `shipped` orders require further investigation to determine whether these values represent specific transaction characteristics or inconsistencies in payment recording.

## Revenue Impact Assessment

### Objective

Estimate the financial impact of invalid payment records by comparing their contribution to total recorded revenue.

This assessment helps determine whether excluding invalid payments would materially affect business reporting.

In [46]:
invalid_revenue = (
    invalid_payment["payment_value"]
    .sum()
)

total_revenue = (
    payments["payment_value"]
    .sum()
)

print(f"Invalid Revenue : {invalid_revenue:.2f}")

print(f"Revenue Impact : {(invalid_revenue / total_revenue) * 100:.4f}%")

Invalid Revenue : 0.00
Revenue Impact : 0.0000%


### Observation

- A total of **9** payment records with `payment_value <= 0` were identified; however, the combined `payment_value` of these records is **0.00**.
- This value represents **0.0000%** of the overall `payment_value`, indicating that these records do not make a material contribution to the total recorded transaction value.
- Although the impact on the total transaction value is relatively insignificant, these records should still be examined based on `payment_type` and `order_status` to determine whether the condition represents a specific transaction characteristic or a recording inconsistency.

## Business Impact Assessment

### Objective

Evaluate whether invalid payment records materially affect customer-level business metrics.

This assessment compares customer spending and transaction behaviour with and without invalid payment records to determine whether the anomaly influences downstream analytics such as revenue, customer lifetime value, and churn modelling.

In [47]:
valid_payment = payments.loc[
    payments["payment_value"] > 0
]

summary = pd.DataFrame({
    "Metric": [
        "Total Payment Records",
        "Valid Payment Records",
        "Invalid Payment Records",
        "Invalid Percentage (%)"
    ],
    "Value": [
        len(payments),
        len(valid_payment),
        len(invalid_payment),
        round(
            len(invalid_payment) /
            len(payments) * 100,
            4
        )
    ]
})

summary

,Metric,Value
0,Total Payment Records,103886.0000
1,Valid Payment Records,103877.0000
2,Invalid Payment Records,9.0000
3,Invalid Percentage (%),0.0087


### Observation

- The dataset contains **103,886** payment records, of which **103,877** are classified as valid based on `payment_value > 0`.
- Only **9** records are classified as invalid, representing **0.01%** of the total payment records.
- Therefore, the proportion of payment records with non-positive values is extremely small compared to the overall dataset.
- Based on the record-level proportion, the issue is unlikely to materially affect analyses involving the overall payment dataset, although the affected records should still be reviewed before final data cleaning.

## Sample Record Investigation

### Objective

Review individual invalid payment records to understand their characteristics and identify possible explanations behind the detected anomalies.

This qualitative inspection complements the quantitative analysis performed in previous sections.

In [48]:
payment_status[
    [
        "order_id",
        "payment_type",
        "payment_sequential",
        "payment_installments",
        "payment_value",
        "order_status"
    ]
].head(20)

,order_id,payment_type,payment_sequential,payment_installments,payment_value,order_status
0,8bcbe01d44d147f901cd3192671144db,voucher,4,1,0.0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,voucher,14,1,0.0,shipped
2,6ccb433e00daae1283ccc956189c82ae,voucher,4,1,0.0,delivered
3,4637ca194b6387e2d538dc89b124b0ee,not_defined,1,1,0.0,canceled
4,00b1cb0320190ca0daa2c88b35206009,not_defined,1,1,0.0,canceled
5,45ed6e85398a87c253db47c2d9f48216,voucher,3,1,0.0,delivered
6,fa65dad1b0e818e3ccc5cb0e39231352,voucher,13,1,0.0,shipped
7,c8c528189310eaa44a745b8d9d26908b,not_defined,1,1,0.0,canceled
8,b23878b3e8eb4d25a158f57d96331b18,voucher,4,1,0.0,delivered


### Observation

- All **9** payment records identified as non-positive have a `payment_value` of **0.00**, indicating that no negative payment values are present in the dataset.
- **6** records are associated with `payment_type = voucher`, while **3** records have `payment_type = not_defined`.
- The voucher-related records are associated with orders in `delivered` and `shipped` status, while all `not_defined` records are associated with `canceled` orders.
- The presence of zero payment values in `voucher` transactions may indicate that the payment was fully covered by a voucher or that the payment was recorded separately from the order value.
- The **3** `not_defined` payment records associated with `canceled` orders require further validation to determine whether the zero payment value represents a valid canceled transaction or incomplete payment recording.

## Multi-Payment Investigation

### Objective

Determine whether invalid payment values occur as standalone payments or as part of multiple payment transactions for the same order.

This assessment helps distinguish between genuinely invalid payments and legitimate edge cases resulting from split payment scenarios.

In [49]:
payment_count = (
    payments
    .groupby("order_id")
    .size()
    .rename("payment_records")
)

multi_payment = (
    invalid_payment
    .merge(
        payment_count,
        on="order_id",
        how="left"
    )
)

multi_payment[
    "payment_records"
].value_counts().sort_index()

payment_records
1     3
3     1
4     3
29    2
Name: count, dtype: int64

### Observation

- The **9** non-positive payment records are associated with orders having different numbers of payment records, ranging from **1 to 29 records per order**.
- **3 orders** have only **1 payment record**, while **3 orders** have **4 payment records** and **2 orders** have **29 payment records**.
- The presence of multiple payment records for the same order indicates that a zero payment value may not necessarily represent the complete payment activity of the order.
- In particular, the **2 orders with 29 payment records** require further investigation because the non-positive payment may represent one component of a multi-record payment structure rather than the total payment for the order.

## Order Value vs Payment Value Consistency 

### Objective

Compare the total order value derived from `price` and `freight_value` with the total recorded `payment_value` for orders containing non-positive payment records.

This assessment aims to determine whether zero payment values represent incomplete payment records or valid transaction conditions, particularly for orders involving vouchers or multiple payment records.

In [61]:
# Order Value vs Payment Value Consistency
# ==========================================================

order_value = (
    order_items
    .assign(
        item_value=lambda x:
        x["price"] + x["freight_value"]
    )
    .groupby("order_id", as_index=False)["item_value"]
    .sum()
    .rename(columns={
        "item_value": "order_value"
    })
)

total_payment = (
    payments
    .groupby("order_id", as_index=False)["payment_value"]
    .sum()
    .rename(columns={
        "payment_value": "total_payment_value"
    })
)

payment_consistency = (
    invalid_payment[
        [
            "order_id",
            "payment_type",
            "payment_value"
        ]
    ]
    .merge(
        orders[
            [
                "order_id",
                "order_status"
            ]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        order_value,
        on="order_id",
        how="left"
    )
    .merge(
        total_payment,
        on="order_id",
        how="left"
    )
)

payment_consistency

,order_id,payment_type,payment_value,order_status,order_value,total_payment_value
0,8bcbe01d44d147f901cd3192671144db,voucher,0.0,delivered,74.16,74.16
1,fa65dad1b0e818e3ccc5cb0e39231352,voucher,0.0,shipped,457.99,457.99
2,6ccb433e00daae1283ccc956189c82ae,voucher,0.0,delivered,122.04,122.04
3,4637ca194b6387e2d538dc89b124b0ee,not_defined,0.0,canceled,NaN,0.00
4,00b1cb0320190ca0daa2c88b35206009,not_defined,0.0,canceled,NaN,0.00
5,45ed6e85398a87c253db47c2d9f48216,voucher,0.0,delivered,71.14,71.14
6,fa65dad1b0e818e3ccc5cb0e39231352,voucher,0.0,shipped,457.99,457.99
7,c8c528189310eaa44a745b8d9d26908b,not_defined,0.0,canceled,NaN,0.00
8,b23878b3e8eb4d25a158f57d96331b18,voucher,0.0,delivered,171.57,171.57


### Observation

- The investigation shows that the **6** non-positive payment records associated with `voucher` have a `total_payment_value` equal to their corresponding `order_value`.
- This indicates that the zero-value voucher records do not represent unpaid orders, as the overall payment value at the order level remains consistent with the order value.
- The remaining **3** non-positive payment records are associated with `canceled` orders and have a `total_payment_value` of **0.00**. No corresponding `order_items` records were found for these orders.
- Therefore, the **9** non-positive payment records should not be treated as erroneous solely based on their individual `payment_value = 0.00`.

## Payment Installments Anomaly Investigation

### Objective

Investigate payment records with invalid `payment_installments` values by examining their overlap with `payment_value` anomalies, `order_status`, and `payment_type`.

This assessment aims to determine whether non-positive installment values are associated with specific transaction characteristics or represent potential data recording inconsistencies.

In [64]:
print("Payment Installments Anomaly Investigation")
print("-" * 50)

invalid_installments = payments.loc[payments["payment_installments"] <= 0].copy()

print(f"\nTotal records: {len(invalid_installments)}")
display(invalid_installments)

# Cek overlap dengan payment_value anomaly
overlap = invalid_installments["order_id"].isin(invalid_payment["order_id"])
print(f"\nOverlap dengan payment_value <= 0 anomaly: {overlap.sum()} dari {len(invalid_installments)} baris")

# Cek order_status
installments_status = invalid_installments.merge(
    orders[["order_id", "order_status"]], on="order_id", how="left"
)
print("\nOrder Status Distribution:")
print(installments_status["order_status"].value_counts())

print("\nPayment Type Distribution:")
print(invalid_installments["payment_type"].value_counts())

Payment Installments Anomaly Investigation
--------------------------------------------------

Total records: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94



Overlap dengan payment_value <= 0 anomaly: 0 dari 2 baris

Order Status Distribution:
order_status
delivered    2
Name: count, dtype: int64

Payment Type Distribution:
payment_type
credit_card    2
Name: count, dtype: int64


In [66]:
print("Installments Pattern Check: Sequential Payment Records")
print("-" * 60)

# Cek distribusi installments untuk SEMUA payment_sequential >= 2 (bukan cuma yang anomali)
sequential_2plus = payments[payments["payment_sequential"] >= 2]

print(f"\nTotal payment records dengan sequential >= 2: {len(sequential_2plus)}")
print("\nDistribusi payment_installments untuk sequential >= 2:")
print(sequential_2plus["payment_installments"].value_counts().sort_index())

print("\nDistribusi payment_type untuk sequential >= 2:")
print(sequential_2plus["payment_type"].value_counts())

Installments Pattern Check: Sequential Payment Records
------------------------------------------------------------

Total payment records dengan sequential >= 2: 4526

Distribusi payment_installments untuk sequential >= 2:
payment_installments
0        2
1     4310
2       53
3       39
4       32
5       18
6       16
7        7
8       26
10      23
Name: count, dtype: int64

Distribusi payment_type untuk sequential >= 2:
payment_type
voucher        4154
credit_card     319
debit_card       52
boleto            1
Name: count, dtype: int64


In [67]:
payment_consistency["payment_type"].value_counts()

payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

In [68]:
invalid_payment["order_id"].nunique()

8

### Observation

- A total of **2** payment records were identified with `payment_installments = 0`, and both were associated with `credit_card` payments and `delivered` orders.
- Among **4,526** payment records with `payment_sequential >= 2`, **4,310** records had `payment_installments = 1`, while only **2** records had `payment_installments = 0`. This indicates that zero installment values are extremely rare among sequential payment records.
- The **2** records with `payment_installments = 0` did not overlap with the **9** records having `payment_value <= 0`, and both had positive `payment_value` values of **58.69** and **129.94**.
- The findings suggest that the `payment_installments = 0` records are isolated cases rather than a broader pattern in sequential payments. However, they should still be reviewed as potential recording inconsistencies in `payment_installments`.

## Summary of Findings Section 2

### Scope of the Issue

The payment quality assessment identified two separate types of anomalies that do not overlap with each other:

1. `payment_value <= 0` — **9 out of 103,886 payment records (0.01%)**
2. `payment_installments <= 0` — **2 out of 103,886 payment records**, with no overlap with the `payment_value` anomalies.

### Payment Value Anomaly — Detailed Findings

**Distribution:** Of the 9 anomalous records, **6 records are associated with `payment_type = voucher`**, while the remaining **3 records are classified as `not_defined`**.

**Order status:** 4 records are associated with `delivered` orders, 3 with `canceled` orders, and 2 with `shipped` orders. All 9 records have a `payment_value` of exactly **0.00**; no negative values were found in the dataset.

**Multi-payment structure:** The orders associated with these 9 anomalous records have varying numbers of payment records per order, ranging from **1 to 29 payment records** (3 orders with 1 record, 1 order with 3 records, 3 orders with 4 records, and 2 orders with 29 records). This indicates that a `payment_value = 0` on an individual record does not necessarily mean that the order was unpaid. Instead, it may represent only one component of a multi-record payment structure.

**Root cause (order-level consistency check):** This is the most important finding. By comparing the total `payment_value` at the order level with the `order_value` calculated from `price + freight_value` in `order_items`, the following was confirmed:

- **6 voucher records:** The total payment at the order level remains consistent with the corresponding `order_value`. The voucher is recorded as a separate payment record with a value of 0, while the actual payment amount is already captured in other payment records associated with the same order.
- **3 `not_defined` records associated with canceled orders:** No `order_items` were found for these orders, which is consistent with orders being canceled before any item-level transaction took place. Therefore, `payment_value = 0` represents a valid business condition rather than missing or corrupted data.

**Conclusion:** All **9 `payment_value` anomalies are confirmed to be business-valid records rather than data quality errors**.

### Installments Anomaly — Detailed Findings

**Characteristics:** The 2 records with `payment_installments = 0` both have `payment_sequential = 2` (the second payment record within the order), `payment_type = credit_card`, and `order_status = delivered`. Both records also have positive and reasonable `payment_value` amounts of **58.69 and 129.94**.

**Pattern check:** To determine whether this represents a systematic pattern or a genuine anomaly, all **4,526 payment records with `payment_sequential >= 2`** were examined. The results show that **95% (4,310 records) have `installments = 1`**, making it the dominant value, while only **2 records have `installments = 0`**. This indicates that the two records are highly uncommon within the payment structure and should be treated as an anomaly requiring further consideration.

### Additional Section — Relationship Validation

**Objective**

To ensure the integrity of relationships between tables by checking the validity of foreign keys, identifying orphan records, and assessing relationship cardinality. The validation covers the relationships between customer–order, order–order item, order–payment, order–review, product–order item, and seller–order item. The results are used to ensure that the dataset has consistent relationships before proceeding to the next stage of analysis.

In [78]:
print("=" * 70)
print("03 — Relationship Validation")
print("=" * 70)


# 1. Customer → Order
# ==========================================================

print("\n[1] Customer → Order")

customer_ids = set(customers["customer_id"])

order_customer_missing = orders["customer_id"].isna().sum()
order_customer_orphan = (~orders["customer_id"].isin(customer_ids)).sum()

customer_order_counts = orders.groupby("customer_id")["order_id"].nunique()

print(f"Orders without customer_id : {order_customer_missing:,}")
print(f"Orders with orphan customer : {order_customer_orphan:,}")
print(f"Customers with orders       : {customer_order_counts.size:,}")
print(f"Maximum orders per customer : {customer_order_counts.max():,}")


# ==========================================================
# 2. Order → Order Item
# ==========================================================

print("\n[2] Order → Order Item")

order_ids = set(orders["order_id"])

item_orphan_orders = (~order_items["order_id"].isin(order_ids)).sum()

order_item_counts = order_items.groupby("order_id")["order_item_id"].count()

orders_without_items = len(
    set(orders["order_id"]) - set(order_items["order_id"])
)

print(f"Order items with orphan order : {item_orphan_orders:,}")
print(f"Orders without order items   : {orders_without_items:,}")
print(f"Orders with items             : {order_item_counts.size:,}")
print(f"Maximum items per order       : {order_item_counts.max():,}")


# ==========================================================
# 3. Order → Payment
# ==========================================================

print("\n[3] Order → Payment")

payment_orphan_orders = (~payments["order_id"].isin(order_ids)).sum()

payment_counts = payments.groupby("order_id")["payment_sequential"].count()

orders_without_payment = len(
    set(orders["order_id"]) - set(payments["order_id"])
)

print(f"Payments with orphan order : {payment_orphan_orders:,}")
print(f"Orders without payment    : {orders_without_payment:,}")
print(f"Orders with payment       : {payment_counts.size:,}")
print(f"Maximum payments per order: {payment_counts.max():,}")


# ==========================================================
# 4. Order → Review
# ==========================================================

print("\n[4] Order → Review")

review_orphan_orders = (~reviews["order_id"].isin(order_ids)).sum()

review_counts = reviews.groupby("order_id")["review_id"].count()

orders_without_review = len(
    set(orders["order_id"]) - set(reviews["order_id"])
)

orders_one_review = (review_counts == 1).sum()
orders_multiple_reviews = (review_counts > 1).sum()

print(f"Reviews with orphan order   : {review_orphan_orders:,}")
print(f"Orders without review      : {orders_without_review:,}")
print(f"Orders with one review     : {orders_one_review:,}")
print(f"Orders with multiple reviews: {orders_multiple_reviews:,}")
print(f"Maximum reviews per order   : {review_counts.max():,}")


# ==========================================================
# 5. Product → Order Item
# ==========================================================

print("\n[5] Product → Order Item")

product_ids = set(products["product_id"])

product_orphans = (~order_items["product_id"].isin(product_ids)).sum()

print(f"Order items with orphan product : {product_orphans:,}")


# ==========================================================
# 6. Seller → Order Item
# ==========================================================

print("\n[6] Seller → Order Item")

seller_ids = set(sellers["seller_id"])

seller_orphans = (~order_items["seller_id"].isin(seller_ids)).sum()

print(f"Order items with orphan seller : {seller_orphans:,}")

review_counts = reviews.groupby("order_id")["review_id"].count()

03 — Relationship Validation

[1] Customer → Order
Orders without customer_id : 0
Orders with orphan customer : 0
Customers with orders       : 99,441
Maximum orders per customer : 1

[2] Order → Order Item
Order items with orphan order : 0
Orders without order items   : 775
Orders with items             : 98,666
Maximum items per order       : 21

[3] Order → Payment
Payments with orphan order : 0
Orders without payment    : 1
Orders with payment       : 99,440
Maximum payments per order: 29

[4] Order → Review
Reviews with orphan order   : 0
Orders without review      : 768
Orders with one review     : 98,126
Orders with multiple reviews: 547
Maximum reviews per order   : 3

[5] Product → Order Item
Order items with orphan product : 0

[6] Seller → Order Item
Order items with orphan seller : 0


In [79]:
customers_without_orders = customers[
    ~customers["customer_id"].isin(orders["customer_id"])
]["customer_id"].nunique()
print(f"Customers without any order: {customers_without_orders}")

Customers without any order: 0


### Observation

- **Customer → Order:** No missing or orphan `customer_id` values were found. All **99,441 customers with orders** have valid references, and the maximum number of orders per customer is **1**. This indicates a **one-to-one relationship** between customers and orders within the dataset.

- **Order → Order Item:** No orphan `order_id` values were found in the order item table. However, **775 orders do not have any order items**, while **98,666 orders have at least one item**. The maximum number of items per order is **21**, indicating that an order can contain multiple items.

- **Order → Payment:** No orphan `order_id` values were found in the payment table. Only **1 order does not have a payment record**, while **99,440 orders have at least one payment record**. The maximum number of payment records per order is **29**, indicating that the relationship can be **one-to-many**.

- **Order → Review:** No orphan `order_id` values were found in the review table. However, **768 orders do not have a review**, which is expected because not every order necessarily receives a review. The remaining orders have review records, with some orders potentially having multiple reviews.

- **Product → Order Item:** No orphan `product_id` values were found. This indicates that all products referenced in `order_items` exist in the `products` table.

- **Seller → Order Item:** No orphan `seller_id` values were found. This indicates that all sellers referenced in `order_items` exist in the `sellers` table.

**Overall**, the validation shows that the dataset has **consistent referential integrity**, with no orphan foreign-key references across the checked relationships. The main exceptions are records where related data is absent, such as orders without order items, payments, or reviews. These cases do not necessarily indicate data quality issues and may represent valid business scenarios.